# AICellID — Blood Cell Classifier (Colab)

Fine-tunes ResNet18 on the split blood cell dataset.

## Before you run
1. **Enable GPU:** Runtime → Change runtime type → T4 GPU (free).
2. **Upload `pbc_split.zip`** (~267 MB) to your Google Drive at `MyDrive/AICellID/pbc_split.zip`.
3. Run the cells in order. The trained checkpoint will be saved to `MyDrive/AICellID/cell_classifier.pt` — download it and put it in `backend/models/` in the repo.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Copy and unzip dataset to local Colab disk
Reading from local disk is much faster than reading directly from Drive.

In [ ]:
!cp /content/drive/MyDrive/AICellID/pbc_split.zip /content/pbc_split.zip
!unzip -q /content/pbc_split.zip -d /content/data
!ls /content/data

## 3. Imports and config

In [ ]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

DATA_DIR = '/content/data'
BATCH = 64
EPOCHS = 5
LR = 1e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## 4. Datasets and dataloaders

In [ ]:
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(f'{DATA_DIR}/train', transform=train_tf)
val_ds = datasets.ImageFolder(f'{DATA_DIR}/val', transform=val_tf)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
val_dl = DataLoader(val_ds, batch_size=BATCH, num_workers=2, pin_memory=True)
print('classes:', train_ds.classes)
print('train:', len(train_ds), ' val:', len(val_ds))

## 5. Model

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(train_ds.classes))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

## 6. Train

In [ ]:
best_acc = 0.0
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for x, y in train_dl:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    train_loss /= len(train_ds)

    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in val_dl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            correct += (model(x).argmax(1) == y).sum().item()
    acc = correct / len(val_ds)
    print(f'epoch {epoch+1}: train_loss={train_loss:.4f}  val_acc={acc:.4f}')
    best_acc = max(best_acc, acc)
print(f'best val acc: {best_acc:.4f}')

## 7. Save checkpoint to Drive

In [ ]:
import os
os.makedirs('/content/drive/MyDrive/AICellID', exist_ok=True)
ckpt_path = '/content/drive/MyDrive/AICellID/cell_classifier.pt'
torch.save({'state_dict': model.state_dict(), 'classes': train_ds.classes}, ckpt_path)
print('saved:', ckpt_path)